# End-to-End Validation With Effective Context
This is the canonical multi-table runner. It applies current and trusted context, discovers relationships, profiles data, performs migration or BigQuery-only validation, runs RCA, and writes consolidated reports.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import load_app_config
from dq_agent.context_store import read_context, write_context_proposals
from dq_agent.context_utils import (
    build_context_proposals, build_effective_inputs, configure_workflow_logging,
    normalize_input_context, context_workflow_paths, logged_step, rank_relevant_context,
    workflow_reviews_to_records, write_approval_workbook,
)
from dq_agent.reporting import write_json
from dq_agent.workflow import DQWorkflow

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
VALIDATION_RUN_ID = f'validation_{RUN_ID}'
RUN_VALIDATION = False  # Requires working Databricks/BigQuery warehouse credentials
STOP_AFTER = 'reports'  # metadata, inference, rules, execution, rca, or reports
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'RETRIEVE_RELEVANT_CONTEXT'):
    current = normalize_input_context(config, RUN_ID, logger)
    trusted = read_context(config, logger=logger) if config.project.context_store.enabled else pd.DataFrame()
    relevant = rank_relevant_context(current, trusted, config.project.confidence.review)
display(relevant)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'APPLY_CONTEXT_PRECEDENCE'):
    effective_inputs = build_effective_inputs(current, trusted)
    write_json(paths['output'] / 'effective_context.json', effective_inputs)
    print('Tables in this run:', [row['pair_id'] for row in effective_inputs['table_mappings']])
display(pd.DataFrame([
    {'context_id': key, **value}
    for key, value in effective_inputs['business_context']['tables'].items()
]))

In [ ]:
manifest = None
if RUN_VALIDATION:
    with logged_step(logger, paths['checkpoint'], 'RUN_END_TO_END_VALIDATION', run_id=VALIDATION_RUN_ID, stop_after=STOP_AFTER):
        manifest = DQWorkflow(ROOT, effective_inputs=effective_inputs).run(
            run_id=VALIDATION_RUN_ID, stop_after=STOP_AFTER
        )
    validation_output = config.path(config.project.outputs_dir) / VALIDATION_RUN_ID
    display(pd.DataFrame(manifest['tables']))
    print('Run status:', manifest['status'])
    print('Output directory:', validation_output)
    print('Reports:', [str(path) for path in sorted(validation_output.glob('*.xlsx'))])
else:
    print('Validation skipped. Inspect effective context above, then set RUN_VALIDATION=True when warehouses are available.')

In [ ]:
if manifest:
    validation_output = config.path(config.project.outputs_dir) / VALIDATION_RUN_ID
    checkpoint_file = validation_output / 'checkpoint.json'
    event_file = validation_output / 'events.jsonl'
    if checkpoint_file.exists():
        print('Latest workflow checkpoint:')
        display(json.loads(checkpoint_file.read_text(encoding='utf-8')))
    if event_file.exists():
        recent_events = [json.loads(line) for line in event_file.read_text(encoding='utf-8').splitlines()[-20:]]
        print('Most recent workflow events:')
        display(pd.DataFrame(recent_events))
    print('Developer log:', validation_output / 'run.log')

In [ ]:
if manifest:
    proposal_file = Path(manifest['approval_proposals_file'])
    workflow_reviews = json.loads(proposal_file.read_text(encoding='utf-8'))
    with logged_step(logger, paths['checkpoint'], 'WRITE_CONTEXT_STAGING', source=str(proposal_file)):
        inferred_records = workflow_reviews_to_records(workflow_reviews, effective_inputs, config, RUN_ID)
        inferred_proposals = build_context_proposals(inferred_records, trusted, RUN_ID)
        if config.project.context_store.enabled:
            print(write_context_proposals(config, inferred_proposals, logger))
    with logged_step(logger, paths['checkpoint'], 'CREATE_APPROVAL_FILE'):
        if not inferred_proposals.empty:
            approval_file = paths['pending'] / f'agent_inference_approvals_{RUN_ID}.xlsx'
            write_approval_workbook(approval_file, inferred_proposals)
            print('Approval file:', approval_file)
        else:
            print('No unresolved agent inferences.')

## What to inspect
Open `consolidated_report.xlsx` for the overall result. Each table folder contains metadata, inference evidence, generated SQL, relationship candidates, masked row reconciliation, RCA, and `validation_report.xlsx`. Review `run.log`, `events.jsonl`, and `checkpoint.json` when a step fails. Approval still uses the configured BigQuery context store and notebook `02`.